In [3]:
import pandas as pd
import numpy as np
import os

# 1. Muat dataset dari direktori raw
df = pd.read_csv('../data/raw/Telco-Customer-Churn.csv')

# 2. Paksa konversi TotalCharges ke numerik
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 3. Imputasi analitis: Isi nilai NaN dengan 0 karena tenure = 0
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# 4. Validasi kebersihan data
print("Jumlah nilai kosong pada TotalCharges saat ini:", df['TotalCharges'].isnull().sum())

# Koreksi Operasional: Pastikan direktori target eksis sebelum menulis file
output_dir = '../data/processed'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 5. Simpan dataset ke direktori processed
df.to_csv(f'{output_dir}/cleaned_telco.csv', index=False)
print("Data bersih berhasil diekspor tanpa halangan.")

Jumlah nilai kosong pada TotalCharges saat ini: 0
Data bersih berhasil diekspor tanpa halangan.


In [7]:
!git config --global user.email "isaknewton17@gmail.com"
!git config --global user.name "AdityaAulia"
!git remote set-url origin https://<TOKEN_ANDA>@github.com/AdityaAulia/churn-business-valuation.git
!git add .
!git commit -m "feat: inisialisasi infrastruktur data pipeline dan pembersihan awal data totalcharges"
!git push origin main

The system cannot find the file specified.


[main af4c9fa] feat: inisialisasi infrastruktur data pipeline dan pembersihan awal data totalcharges
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 notebooks/01_data_cleaning.ipynb


To https://github.com/AdityaAulia/churn-business-valuation.git
   cc9eb98..af4c9fa  main -> main


In [8]:
# Memuat data dari folder processed untuk memastikan kita menggunakan data bersih
import pandas as pd
df = pd.read_csv('../data/processed/cleaned_telco.csv')

# Drop customerID karena tidak memiliki nilai prediktif (High Cardinality)
df = df.drop(columns=['customerID'])

# Filter kolom dengan tipe data object
categorical_cols = df.select_dtypes(include=['object']).columns
print("Variabel Kategorikal yang harus ditransformasi:")
print(list(categorical_cols))
print(f"\nTotal variabel kategorikal: {len(categorical_cols)}")

Variabel Kategorikal yang harus ditransformasi:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']

Total variabel kategorikal: 16


In [9]:
# 1. Label Encoding untuk variabel binarian (Manual Mapping)
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']

# Memastikan nama kolom sesuai casing dataset (Python bersifat case-sensitive)
for col in binary_cols:
    if col in df.columns:
        # Mengubah nilai pertama menjadi 0 dan nilai kedua menjadi 1
        df[col] = df[col].map({df[col].unique()[0]: 0, df[col].unique()[1]: 1})

# 2. One-Hot Encoding untuk variabel multikelas tersisa
# drop_first=True digunakan untuk menghindari jebakan multikolinieritas (Dummy Variable Trap)
df_encoded = pd.get_dummies(df, drop_first=True)

# 3. Konversi seluruh nilai boolean hasil get_dummies menjadi integer (0/1)
df_encoded = df_encoded.astype(int)

# 4. Inspeksi hasil transformasi
print("Dimensi dataset setelah Rekayasa Fitur:", df_encoded.shape)
df_encoded.head()

Dimensi dataset setelah Rekayasa Fitur: (7043, 31)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,0,0,1,0,0,29,29,0,...,0,0,0,0,0,0,0,0,1,0
1,1,0,1,0,34,1,1,56,1889,0,...,0,0,0,0,0,1,0,0,0,1
2,1,0,1,0,2,1,0,53,108,1,...,0,0,0,0,0,0,0,0,0,1
3,1,0,1,0,45,0,1,42,1840,0,...,1,0,0,0,0,1,0,0,0,0
4,0,0,1,0,2,1,0,70,151,1,...,0,0,0,0,0,0,0,0,1,0


In [11]:
# Menyimpan dataset siap latih
df_encoded.to_csv('../data/processed/features_telco.csv', index=False)
print("Data fitur berhasil diekspor ke ../data/processed/features_telco.csv")

Data fitur berhasil diekspor ke ../data/processed/features_telco.csv
